# Validacion SQL internacional
Este notebook documenta la comparacion entre el consolidado Parquet y `tsi_db` mediante los resultados de validacion registrados.

In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS = PROJECT_ROOT / 'data' / '03_analysis_output' / 'validacion_sql'
state = json.loads((RESULTS / 'estado.json').read_text(encoding='utf-8'))
print(state)

{'estado': 'completa', 'metodo': 'validacion_manual_sql_por_fuente', 'observaciones_parquet': 41770880, 'observaciones_mysql': 41770880, 'diferencia_total': 0, 'fuentes': {'istanbul_traffic_index': 3332, 'madrid_traffic_points': 12512133, 'metr_la': 7094304, 'pems_bay': 16937700, 'sao_paulo_cet': 5223411}}


In [2]:
source_counts = pd.Series(state['fuentes'], name='observaciones').rename_axis('fuente').reset_index()
source_counts['porcentaje'] = source_counts['observaciones'] / source_counts['observaciones'].sum() * 100
display(source_counts.sort_values('observaciones', ascending=False))
assert state['observaciones_parquet'] == state['observaciones_mysql']
assert state['diferencia_total'] == 0
assert source_counts['observaciones'].sum() == state['observaciones_mysql']
print('controles=aprobados')

,fuente,observaciones,porcentaje
3,pems_bay,16937700,40.549062
1,madrid_traffic_points,12512133,29.954200
2,metr_la,7094304,16.983851
4,sao_paulo_cet,5223411,12.504910
0,istanbul_traffic_index,3332,0.007977


controles=aprobados


In [3]:
comparison_path = RESULTS / 'comparacion_por_fuente.csv'
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    display(comparison)
else:
    print('No existe comparacion_por_fuente.csv; se conserva la validacion por conteo y fuente.')

No existe comparacion_por_fuente.csv; se conserva la validacion por conteo y fuente.


## Manual SQL
Las consultas de `06_consultas_validacion.sql` permiten repetir el control desde MySQL Workbench. Ejecutarlas una por una porque el conteo agrupado recorre la tabla de hechos completa.